# 51 Peg: Exo-Striker from Python

**Example provided by Trifon Trifonov with the aid of CODEX; use with caution.**

Fit the bundled ELODIE and HARPS velocities with a circular single-planet model.
The instruments have very different zero points, so each receives its own offset
and jitter. The fixed stellar mass of 1.0 solar mass is an illustrative tutorial
assumption. A single Keplerian planet needs no planet–planet dynamical comparison.

This tutorial follows the fitting → priors → nested sampling → diagnostics →
CSV exports structure of `Alans_NB/NS_nu_Oph_eng_v3.ipynb`, updated for the current
Exo-Striker API. It starts from the repository data files; no old session is needed.
The matching script is `51Peg_RV.py`. Shared plotting/export utilities live in
`example_tools.py`.

Run the cells in order. Nested sampling is optional and disabled initially, so
**Run All** first produces a best-fit analysis. Enable it explicitly below for
posterior uncertainties and evidence; a full run can take minutes to hours.

## 1. Setup and paths

Use a Python environment with this checkout's Exo-Striker dependencies installed
(`python -m pip install -e .` from the repository root). The first import may
compile the native RV backend, requiring a working Fortran compiler, Meson and Ninja.
No external LaTeX installation or downloads of observational data are needed.
Open Jupyter from the repository root or `examples/`; paths are resolved before
importing Exo-Striker, because older releases change the working directory.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / 'exostriker' / 'datafiles').is_dir()
             and (p / 'examples' / 'example_tools.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or examples directory.')
sys.path.insert(0, str(ROOT / 'examples'))
from example_tools import (DATA, EXAMPLES, rv, np, plt, uniform_bound,
                           check_priors, fit_rvs, plot_rvs, finish, command_line)
from example_tools import configure_sampling, export_posterior, save_session

In [ ]:
TARGET = '51Peg'
OUTPUT = EXAMPLES / 'results' / TARGET
RUN_NESTED = False   # Set True to run the optional posterior analysis.
NS_THREADS = 1       # Start with one process; increase deliberately for long runs.
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Data:', DATA)
print('Outputs:', OUTPUT)

## 2. Data, starting model, and explicit priors

The functions below show every starting value and prior. Parameter slots are
zero-based. Uniform bounds are set explicitly and Gaussian/log-uniform priors
are disabled for those parameters. These are tutorial choices, not evidence for
a particular prior. For model comparisons, choose justified priors before the
analysis and keep shared priors consistent across models.

In [ ]:
def set_priors(fit):
    # Circular orbit: e and omega stay fixed, avoiding an undefined omega at e=0.
    for key, limits in {'K': (1., 100.), 'P': (4.22, 4.24), 'M0': (0., 360.)}.items():
        uniform_bound(fit, key, 0, limits)
    for i, limits in enumerate(((-34000., -32500.), (-500., 500.))):
        uniform_bound(fit, 'rvoff', i, limits, plural=True)
        uniform_bound(fit, 'jitt', i, (0., 50.), plural=True)

In [ ]:
def build_fit():
    fit = rv.signal_fit(name='51 Peg — circular RV orbit')
    fit.init_fit = False
    fit.stellar_mass = fit.params.stellar_mass = 1.0  # illustrative fixed mass [Msun]
    for filename in ('51Peg_ELODIE.dat', '51Peg_HARPS.dat'):
        data = np.loadtxt(DATA / filename)
        # The two files have very different velocity zero points, in m/s.
        fit.add_dataset(filename, str(DATA / filename), float(np.median(data[:, 1])),
                        3., usejitter=False)
    fit.add_planet(55., 4.230785, 0., 0., 180., 90., 0., usee=False, usew=False)
    set_priors(fit)
    return fit

fit = build_fit()
print("Reference epoch [JD]:", fit.epoch)
check_priors(fit)

## 3. Optimize the model

The starting period is 4.230785 days. An initial Levenberg–Marquardt fit refines
that period before likelihood Simplex frees the instrument jitters. This matters
because the native Simplex period perturbation can cross many aliases over the
long observing baseline. Eccentricity and argument of periastron are fixed at
zero; the free phase is the mean anomaly at `fit.epoch`. This is a circular-model
example, not a measurement that the eccentricity is exactly zero.

Covariance estimates from optimization are not a substitute for the posterior intervals below.

In [ ]:
def optimize(fit):
    # Refine the known short period with LM before likelihood Simplex.
    # A 5% period perturbation spans many aliases over this long baseline.
    fit.fitting(minimize_fortran=True, minimize_loglik=False,
                outputfiles=[1, 1, 1], amoeba_starts=5, fortran_kill=30)
    for i in range(2):
        fit.jitt_use[i] = True
    fit_rvs(fit, starts=10)
    check_priors(fit)
    return fit

fit = optimize(fit)

## 4. Inspect and save the best fit

The figures show the fitted model and residuals. RV error bars include each
instrument's fitted jitter in quadrature; phase plots remove the other planet's
signal. For WASP-6, both the full time series and the folded transit are shown.
The `.ses` output can be reopened in the Exo-Striker GUI. Re-running this notebook
replaces files of the same name inside `OUTPUT`; change that directory to keep
separate analyses.

In [ ]:
plot_rvs(fit, OUTPUT, TARGET)
save_session(fit, OUTPUT, TARGET)
plt.show()

## 5. Check the sampler parameter space

Inspect every free parameter and its bounds. Sampling uses the same physical
model and fixed assumptions as the final fit above. Starting outside the bounds
is an error; a value near a boundary deserves investigation. Do not silently
shrink priors around this fitted result to accelerate evidence calculations.

In [ ]:
check_priors(fit)

## 6. Optional nested sampling

Dynamic nested sampling uses `rwalk` proposals and multi-ellipsoid bounds.
`nlive = live_points_fact × number of free parameters`; `stop_crit=0.1` controls
the initial evidence tolerance. `ns_pfrac=1.0` allocates work for the posterior.
The iteration and likelihood-call caps are disabled by default. The settings
below are starting choices; convergence must be checked for each analysis.

The current library keeps equal-weight `samples` and `logl` aligned and retains
raw nested samples in `samples_ns` and raw likelihoods in `logl_ns`. The historical
weight-alignment warning in the ν Oph notebook no longer describes this version.
Keep `ns_save_sampler=True` to retain the arrays needed for diagnostics and export.
Scripts protect execution with `if __name__ == '__main__'` for multiprocessing;
use `NS_THREADS=1` initially in notebooks.

In [ ]:
configure_sampling(fit, threads=NS_THREADS)
print('Free parameters:', len(fit.par_for_mcmc))
print('Initial live points:', fit.live_points_fact * len(fit.par_for_mcmc))
print('Run nested sampling:', RUN_NESTED)
if RUN_NESTED:
    fit = rv.run_nestsamp(fit)
    if fit.bound_error:
        raise RuntimeError(fit.bound_error_msg)
else:
    print('Skipped. Set RUN_NESTED=True and rerun this cell to sample.')

## 7. Diagnostics, posterior exports, and corner plot

Raw nested-sampling rows carry unequal weights. The export utility normalizes
`logwt`, computes **ESS = 1 / sum(weight²)** and uses systematic resampling with
seed 42, applying the same row indices to parameter vectors and their `logL`.
It prints iteration/call counts, efficiency, evidence and evidence uncertainty.
Low ESS (the helper flags <200) is a warning; a larger ESS alone does not prove
convergence. Check repeat runs, posterior structure, integration settings where
relevant, and stopping criteria before interpreting evidence or uncertainties.

Outputs include an equal-weight posterior CSV, raw samples with weights and
likelihoods, median/68.3% interval summary CSV, and PNG/PDF corner plots. Column
indices and original labels identify parameters without assuming a packing order.
The maximum-likelihood vector comes from the raw aligned arrays. Evidence is
conditional on the chosen likelihood, fixed assumptions, and prior volume.
The updated session retains the sampler for subsequent inspection.

In [ ]:
if RUN_NESTED:
    posterior = export_posterior(fit, OUTPUT, TARGET)
    save_session(fit, OUTPUT, TARGET + '_NS')
    plt.show()
else:
    print('No posterior exported: nested sampling has not been run.')

## Attribution

**These examples are provided by Trifon Trifonov with the aid of CODEX; use with caution.** Their organization follows the
ν Oph workflow in Alan's notebooks. The observations are the files bundled with
Exo-Striker; this attribution identifies the provider of the examples and does
not assert new ownership of the observational data.